# 歌词分词，词性标注

In [1]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [ ]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [ ]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [4]:
import re
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/hanlp/v21/redirect', auth="699691e7eaf61a3aca90d7b8", language='zh')

def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0


def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []
    
    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')
    
    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']
    
    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()
    
    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):
            
            word = word.strip()
            
            # 过滤标点
            if tag == 'w' or not word:
                continue
            
            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]
            
            word_pos_counter[(word, tag)] += 1
    
    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })
    
    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)
    
    return results


In [ ]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [88]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            print(i['song_name'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [6]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [84]:
# file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"
# file_path_prefix = "data/liuyuning/"
# file_path_prefix = "data/newyear/"
# file_path_prefix = "data/liyuchun/"
# file_path_prefix = "data/chenyixun/"
# file_path_prefix = "data/renxianqi/"
# file_path_prefix = "data/linjunjie/"
file_path_prefix = "data/sunyanzi/"

In [85]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,5211338,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,001pWERg3vFgg8,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
1,5646,003PSWcB4EDl6P,开始懂了,NaN,孙燕姿,109,001pWERg3vFgg8,我要的幸福,493,004KRIQr1oCp2A,271,976118400,开始懂了,我要的幸福,开始懂了,2000-12-07,2000
2,8145,001xJbdb2MBaaz,遇见,《向左走，向右走》电影主题曲,孙燕姿,109,001pWERg3vFgg8,The Moment,586,002ehzTm0TxXC2,210,1061481600,遇见,The Moment,遇见,2003-08-22,2003
3,100893820,001pHzdz29aG1Z,雨天,NaN,孙燕姿,109,001pWERg3vFgg8,My Story 2006 新歌+精选,837815,000VMnoi2Lgt44,239,1159200000,雨天,My Story 2006 新歌+精选,雨天,2006-09-26,2006
4,213446637,0042ih853gboG4,半句再见,"From ""At Café 6"" / Main Theme Song",孙燕姿,109,001pWERg3vFgg8,半句再见,3972762,003c8Hsp0OrnXY,242,1522339200,半句再见,半句再见,半句再见,2018-03-30,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,491752,004JJW4v2So2tx,作战,NaN,孙燕姿,109,001pWERg3vFgg8,Leave,39480,0033sQMp2BZDJC,218,1021910400,作战,Leave,作战,2002-05-21,2002
126,5211227,002lRwn62glwFS,终于,NaN,孙燕姿,109,001pWERg3vFgg8,孙燕姿同名专辑,8707,002UZ9ob4Ecg0S,270,960393600,终于,孙燕姿同名专辑,终于,2000-06-08,2000
127,491779,003rGQse4RcMpr,Silent All These Years,NaN,孙燕姿,109,001pWERg3vFgg8,Start 自选集,39482,000FAIFd0r22m9,255,1012492800,SilentAllTheseYears,Start 自选集,SilentAllTheseYears,2002-02-01,2002
128,5187,000sPOPw1FgNDK,种,NaN,孙燕姿,109,001pWERg3vFgg8,Stefanie,451,003CS0lX1DwEcN,251,1098979200,种,Stefanie,种,2004-10-29,2004


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [89]:
# 词性解析
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
df_word

我怀念的
开始懂了
遇见
雨天
半句再见
天黑黑
我不难过
眼泪成诗
逆光
绿光
原来你什么都不要
当冬夜渐暖
第一天
日落
克卜勒
Honey Honey
样子
坏天气
180度
愚人的国度
The Moment
隐形人
逃亡
我也很想他
Hey Jude
咕叽咕叽
同类
漩涡
我的爱
极美
直来直往
风衣
在,也不见
飘着
神奇
超快感
我要的幸福
尚好的青春
爱情证书
银泰
不是真的爱我
爱情字典
天使的指纹
比较幸福
任性
安宁
了解
平日快乐
一样的夏天
无限大
星期一天气晴我离开你
相信
完美的一天
需要你
天天年年
Venus
时光小偷
祝你开心
风筝
奔
永远
和平
雨还是不停地落下
渴
明天的记忆
余额
世界终结前一天
休止符
眼神
跳舞的梵谷
这个世界
懂事
心愿
害怕
年轻无极限
E-Lover
不能和你一起
很好
彩虹金刚
一起走到
错觉
我想
我很愉快
追
懒得去管
天越亮，夜越黑
梦游
木兰情
世说心语
Radio
Stefanie
我不爱
流浪地图
守护永恒的爱
爱从零开始
天空
学会
难得一见
没有人的方向
关于
真的
是时候 + Hidden Track
慢慢来
Leave Me Alone
围绕
接下来
未完成
浓眉毛
听见
明天晴天
练习
零缺点
太阳底下
另一张脸
随堂测验
橄榄树
累赘
空口言
梦不落
充氧期
Sometimes Love Just Ain't Enough
未知的精彩
不同
中间地带
Leave
作战
终于
Silent All These Years
种
超人类


,song_id,word,pos,freq
0,5211338,我,r,33
1,5211338,的,u,23
2,5211338,是,v,13
3,5211338,记得,v,13
4,5211338,谁,r,12
...,...,...,...,...
11619,206621755,开心,an,1
11620,206621755,体会,v,1
11621,206621755,收回,v,1
11622,206621755,胆怯,a,1


In [90]:
# 重新读取
df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")

In [91]:
df_merged = words_data_merge(df_word_read, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,5211338,我,r,33,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
1,5211338,的,u,23,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
2,5211338,是,v,13,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
3,5211338,记得,v,13,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
4,5211338,谁,r,12,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11619,206621755,开心,an,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017
11620,206621755,体会,v,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017
11621,206621755,收回,v,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017
11622,206621755,胆怯,a,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017


In [92]:
# 过滤中文词
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,5211338,我,r,33,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
1,5211338,的,u,23,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
2,5211338,是,v,13,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
3,5211338,记得,v,13,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
4,5211338,谁,r,12,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,逆光,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11619,206621755,开心,an,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017
11620,206621755,体会,v,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017
11621,206621755,收回,v,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017
11622,206621755,胆怯,a,1,1,004VuJcz2PESBA,超人类,NaN,孙燕姿,109,...,孙燕姿 No.13 作品：跳舞的梵谷,2575698,001In1pS04rvY5,224,1510156800,超人类,孙燕姿 No.13 作品：跳舞的梵谷,超人类,2017-11-09,2017


In [93]:
# 查看歌曲数
df_merged_chn['song_name_pure'].nunique()

126

In [94]:
# 虚拟专辑数据
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
# 只保留120个
df_songs_part = df_songs_part.head(120)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,我怀念的,PART 1,0
1,开始懂了,PART 1,0
2,遇见,PART 1,0
3,雨天,PART 1,0
4,半句再见,PART 1,0
...,...,...,...
115,空口言,PART 12,11
116,梦不落,PART 12,11
117,充氧期,PART 12,11
118,未知的精彩,PART 12,11


In [95]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
# 删除album_order为空的数据
df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year,album_name_raw,album_name,album_order
0,5211338,我,r,33,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007,逆光,PART 1,0.0
1,5211338,的,u,23,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007,逆光,PART 1,0.0
2,5211338,是,v,13,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007,逆光,PART 1,0.0
3,5211338,记得,v,13,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007,逆光,PART 1,0.0
4,5211338,谁,r,12,1,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,...,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007,逆光,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10294,491758,拥有,v,1,1,001gXbnt1lETLJ,不同,NaN,孙燕姿,109,...,220,1021910400,不同,Leave,不同,2002-05-21,2002,Leave,PART 12,11.0
10295,491758,失去,v,1,1,001gXbnt1lETLJ,不同,NaN,孙燕姿,109,...,220,1021910400,不同,Leave,不同,2002-05-21,2002,Leave,PART 12,11.0
10296,491758,想要,v,1,1,001gXbnt1lETLJ,不同,NaN,孙燕姿,109,...,220,1021910400,不同,Leave,不同,2002-05-21,2002,Leave,PART 12,11.0
10297,491758,自由,a,1,1,001gXbnt1lETLJ,不同,NaN,孙燕姿,109,...,220,1021910400,不同,Leave,不同,2002-05-21,2002,Leave,PART 12,11.0


In [96]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [97]:
# 数据查验
songs_n = df_merged_chn[df_merged_chn['pos'] == 'n']['song_name_pure'].unique().tolist()
songs_all = df_merged_chn['song_name_pure'].unique().tolist()
for i in songs_all:
    if i not in songs_n:
        print(i)

# 歌曲数据更新

In [98]:
df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)

df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year,album_name_raw,album_name,album_order
0,5211338,000BO3xy4fd8BJ,我怀念的,《第二回合我爱你（幸运日)》电视剧主题曲,孙燕姿,109,001pWERg3vFgg8,15902,000P3l050Olt27,289,1174492800,我怀念的,逆光,我怀念的,2007-03-22,2007,逆光,PART 1,0.0
1,5646,003PSWcB4EDl6P,开始懂了,NaN,孙燕姿,109,001pWERg3vFgg8,493,004KRIQr1oCp2A,271,976118400,开始懂了,我要的幸福,开始懂了,2000-12-07,2000,我要的幸福,PART 1,0.0
2,8145,001xJbdb2MBaaz,遇见,《向左走，向右走》电影主题曲,孙燕姿,109,001pWERg3vFgg8,586,002ehzTm0TxXC2,210,1061481600,遇见,The Moment,遇见,2003-08-22,2003,The Moment,PART 1,0.0
3,100893820,001pHzdz29aG1Z,雨天,NaN,孙燕姿,109,001pWERg3vFgg8,837815,000VMnoi2Lgt44,239,1159200000,雨天,My Story 2006 新歌+精选,雨天,2006-09-26,2006,My Story 2006 新歌+精选,PART 1,0.0
4,213446637,0042ih853gboG4,半句再见,"From ""At Café 6"" / Main Theme Song",孙燕姿,109,001pWERg3vFgg8,3972762,003c8Hsp0OrnXY,242,1522339200,半句再见,半句再见,半句再见,2018-03-30,2018,半句再见,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,101787307,001UMUsn0cPitA,空口言,NaN,孙燕姿,109,001pWERg3vFgg8,67827,0011TdfZ2J78KH,205,1453737600,空口言,是时候,空口言,2016-01-26,2016,是时候,PART 12,11.0
116,97260,000j8o2r1CXK7U,梦不落,NaN,孙燕姿,109,001pWERg3vFgg8,8188,0019FoJV2aPJiz,226,1128614400,梦不落,完美的一天,梦不落,2005-10-07,2005,完美的一天,PART 12,11.0
117,206621756,002MDDLS0OYo1h,充氧期,NaN,孙燕姿,109,001pWERg3vFgg8,2575698,001In1pS04rvY5,224,1510156800,充氧期,孙燕姿 No.13 作品：跳舞的梵谷,充氧期,2017-11-09,2017,孙燕姿 No.13 作品：跳舞的梵谷,PART 12,11.0
118,5189,004VwbzS4Dj4Lx,未知的精彩,NaN,孙燕姿,109,001pWERg3vFgg8,451,003CS0lX1DwEcN,188,1098979200,未知的精彩,Stefanie,未知的精彩,2004-10-29,2004,Stefanie,PART 12,11.0


In [100]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试

In [101]:
160*4

640